# Ceftriaxone — Adversarial Site-Invariance Diagnostic

**Can we remove the (non-linear) site signal by forcing the resistance MLP's features to be un-informative about site?**

Trains a resistance MLP with a gradient-reversal site head (DANN), sweeps the reversal weight `λ`, and reports:
- resistance BalAcc/AUC (centralized)
- site-probe accuracy of the learned features (want it to drop toward 25%)
- cross-site (A+B+C → D) BalAcc/AUC

Reuses the exact 06-03d data pipeline (same split, downsampling, preprocessing, SEED).

In [ ]:
# ── CONFIG ──
RUN_NAME = "06-03d-AdversarialSite-Diagnostic"
TARGET_RUN = "01-Run"
LAMBDAS = [0.0, 0.1, 0.5, 1.0]
EPOCHS = 80
BATCH_SIZE = 128


In [ ]:
!pip install "flwr[simulation]" maldideepkit maldiamrkit --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEV}")


In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

DRUG_NAME = "Ceftriaxone"

PROJECT_DIR = DRYAD / "Processed/Processing/Analysis/06b-Ceftriaxone-E-coli" / RUN_NAME
RUN_DIR = PROJECT_DIR / TARGET_RUN
OUT_DIR = RUN_DIR / "results"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / DRUG_NAME / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / DRUG_NAME / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / DRUG_NAME / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / DRUG_NAME / "data.csv",
}
SITE_ORDER = ["A", "B", "C", "D"]
print(f"Drug: {DRUG_NAME}  |  Run: {RUN_DIR}")


In [ ]:
# ── Load Ceftriaxone — ALL species ──
raw_data = {}
for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    bin_cols = [c for c in df.columns if c.startswith("bin_")]
    X = df[bin_cols].to_numpy(dtype="float32")
    y = df["label"].to_numpy(dtype="int64")
    species = df["species"].values
    raw_data[site] = (X, y, species)
    n_sp = len(np.unique(species))
    print(f"  Site {site}: {len(y)} samples ({n_sp} species)")
print(f"Total: {sum(len(raw_data[s][1]) for s in SITE_ORDER)}")


In [ ]:
# ── Per-site species-stratified 90/10 split (adaptive test_size) ──
client_train = {}; client_test = {}
species_train = {}; species_test = {}

for site in SITE_ORDER:
    X, y, sp = raw_data[site]
    n = len(y); idx = np.arange(n).reshape(-1,1)
    n_strata = len(set(zip(sp.tolist(), y.tolist())))
    test_size = max(0.10, (n_strata + 1) / n)
    itr, iv, _, _ = stratified_species_drug_split(idx, y, species=sp, test_size=test_size, random_state=SEED)
    itr = itr.flatten().astype(int); iv = iv.flatten().astype(int)
    client_train[site] = (X[itr], y[itr]); client_test[site] = (X[iv], y[iv])
    species_train[site] = sp[itr]; species_test[site] = sp[iv]
    print(f"  Site {site}: train={len(itr)} test={len(iv)} species={len(np.unique(sp[itr]))} (test_size={test_size:.2f})")


In [ ]:
# ── Downsample training data (keep all R, cap S in bad-ratio species) ──
TRAIN_DS_S_PER_R = 10   # global: all training data (bad-ratio species only)

def downsample_keep_idx(y, sp, s_per_r, bad_ratio=0.10, min_n=400, rng=None):
    rng = rng if rng is not None else np.random.default_rng(SEED)
    idx = np.arange(len(y))
    keep = idx[y == 1].tolist()
    for spec in np.unique(sp):
        m = sp == spec
        n_r = int((m & (y == 1)).sum()); n_s = int((m & (y == 0)).sum())
        total = n_r + n_s
        s_pos = idx[m & (y == 0)]
        if n_r > 0 and total > min_n and (n_r / total) < bad_ratio and n_s > n_r * s_per_r:
            keep.extend(rng.choice(s_pos, size=int(round(n_r * s_per_r)), replace=False).tolist())
        else:
            keep.extend(s_pos.tolist())
    return np.sort(np.array(keep, dtype=int))

_rng_ds = np.random.default_rng(SEED)
for site in SITE_ORDER:
    X, y = client_train[site]; sp = species_train[site]
    keep = downsample_keep_idx(y, sp, TRAIN_DS_S_PER_R, rng=_rng_ds)
    client_train[site] = (X[keep], y[keep]); species_train[site] = sp[keep]
    print(f"  Site {site}: kept {len(keep)}/{len(y)} train")


In [ ]:
# ── Per-site preprocessing ──
client_train_pp = {}; client_test_pp = {}
for site in SITE_ORDER:
    X_tr, y_tr = client_train[site]; X_te, y_te = client_test[site]
    state = fit_input_transform(X_tr, "log1p+standardize")
    client_train_pp[site] = (apply_input_transform(X_tr, state), y_tr)
    client_test_pp[site] = (apply_input_transform(X_te, state), y_te)
print("Per-site preprocessing done.")


In [ ]:
# ── Model + Gradient-Reversal layer ──
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

class AdversarialMLP(nn.Module):
    def __init__(self, dropout=0.5):
        super().__init__()
        self.features = nn.Sequential(
            nn.Linear(6000, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout/2),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),
        )
        self.res_head = nn.Linear(128, 2)
        self.site_head = nn.Linear(128, 4)
    def forward(self, x, lambd=0.0):
        z = self.features(x)
        res = self.res_head(z)
        site = self.site_head(GradientReversal.apply(z, lambd))
        return res, site, z

# ── Data assembly ──
X_all = np.concatenate([client_train_pp[s][0] for s in SITE_ORDER]).astype(np.float32)
y_res = np.concatenate([client_train_pp[s][1] for s in SITE_ORDER]).astype(np.int64)
y_site = np.concatenate([np.full(len(client_train_pp[s][0]), i) for i, s in enumerate(SITE_ORDER)]).astype(np.int64)

X_te = np.concatenate([client_test_pp[s][0] for s in SITE_ORDER]).astype(np.float32)
y_te = np.concatenate([client_test_pp[s][1] for s in SITE_ORDER]).astype(np.int64)
y_te_site = np.concatenate([np.full(len(client_test_pp[s][0]), i) for i, s in enumerate(SITE_ORDER)]).astype(np.int64)

# cross-site: train A+B+C, test D (held out)
X_cs_train = np.concatenate([client_train_pp[s][0] for s in ["A","B","C"]]).astype(np.float32)
y_cs_res = np.concatenate([client_train_pp[s][1] for s in ["A","B","C"]]).astype(np.int64)
y_cs_site = np.concatenate([np.full(len(client_train_pp[s][0]), i) for i, s in enumerate(["A","B","C"])]).astype(np.int64)
X_cs_test = client_test_pp["D"][0].astype(np.float32)
y_cs_test = client_test_pp["D"][1].astype(np.int64)
print(f"Pooled train: {len(X_all)}, pooled test: {len(X_te)}, cross-site train: {len(X_cs_train)} -> test D: {len(X_cs_test)}")


In [ ]:
# ── Train + evaluate helpers ──
def train_model(X, y_res, y_site, lambd, epochs=EPOCHS, batch_size=BATCH_SIZE, seed=SEED):
    torch.manual_seed(seed); np.random.seed(seed)
    model = AdversarialMLP().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    Xt = torch.tensor(X, dtype=torch.float32)
    yr = torch.tensor(y_res, dtype=torch.long)
    ys = torch.tensor(y_site, dtype=torch.long)
    crit = nn.CrossEntropyLoss()
    n = len(X)
    model.train()
    for ep in range(epochs):
        perm = torch.randperm(n)
        for i in range(0, n, batch_size):
            idx = perm[i:i+batch_size]
            if len(idx) <= 1:
                continue
            xb = Xt[idx].to(DEV); yrb = yr[idx].to(DEV); ysb = ys[idx].to(DEV)
            res, site, _ = model(xb, lambd)
            loss = crit(res, yrb) + lambd * crit(site, ysb)
            opt.zero_grad(); loss.backward(); opt.step()
    return model

def predict_proba(model, X):
    model.eval()
    with torch.no_grad():
        res, _, _ = model(torch.tensor(X, dtype=torch.float32).to(DEV))
        return F.softmax(res, dim=1).cpu().numpy()[:, 1]

def get_features(model, X):
    model.eval()
    with torch.no_grad():
        _, _, z = model(torch.tensor(X, dtype=torch.float32).to(DEV))
        return z.cpu().numpy()


In [ ]:
# ── Main loop: centralized + cross-site per lambda ──
rows = []
for lambd in LAMBDAS:
    print(f"\n=== lambda={lambd} ===")
    # centralized
    model = train_model(X_all, y_res, y_site, lambd)
    proba = predict_proba(model, X_te)
    res_bal = balanced_accuracy_score(y_te, proba >= 0.5)
    res_auc = roc_auc_score(y_te, proba)
    z_te = get_features(model, X_te)
    probe = LogisticRegression(max_iter=2000).fit(z_te, y_te_site)
    site_probe = probe.score(z_te, y_te_site)
    # cross-site (train A+B+C -> test D)
    model_cs = train_model(X_cs_train, y_cs_res, y_cs_site, lambd)
    proba_cs = predict_proba(model_cs, X_cs_test)
    cs_bal = balanced_accuracy_score(y_cs_test, proba_cs >= 0.5)
    cs_auc = roc_auc_score(y_cs_test, proba_cs)
    rows.append({"lambda": lambd, "res_balacc": res_bal, "res_auc": res_auc,
                 "site_probe_acc": site_probe, "cross_site_balacc": cs_bal, "cross_site_auc": cs_auc})
    print(f"  res BalAcc={res_bal:.4f}  res AUC={res_auc:.4f}  site probe={site_probe:.4f}  "
          f"cross-site BalAcc={cs_bal:.4f}  cross-site AUC={cs_auc:.4f}")

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / "adversarial_site_diagnostic.csv", index=False)
print("\n" + df.to_string(index=False))

# plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.plot(df["lambda"], df["site_probe_acc"], marker='o', color='tab:red', label='site probe (want lower)')
ax.axhline(0.25, color='gray', ls=':', label='chance (4 sites)')
ax.set_xlabel("λ"); ax.set_ylabel("site-probe accuracy"); ax.set_title("Site leakage")
ax.legend(); ax.grid(True, ls='--', alpha=0.5)
ax = axes[1]
ax.plot(df["lambda"], df["res_balacc"], marker='o', label='centralized BalAcc')
ax.plot(df["lambda"], df["cross_site_balacc"], marker='s', label='cross-site BalAcc (A+B+C → D)')
ax.set_xlabel("λ"); ax.set_ylabel("Balanced Accuracy"); ax.set_title("Resistance performance")
ax.legend(); ax.grid(True, ls='--', alpha=0.5)
fig.suptitle(f"{DRUG_NAME} — adversarial site-invariance (λ sweep)", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "adversarial_site_tradeoff.pdf", bbox_inches="tight"); plt.show()


---
**Done.** See `adversarial_site_diagnostic.csv` and `adversarial_site_tradeoff.pdf` in `results/`.

Interpretation:
- `λ=0` = non-adversarial baseline.
- Higher `λ` should push `site_probe_acc` toward 0.25 (site-invariant features).
- `res_balacc` / `cross_site_balacc` should hold or improve.
- If site-probe won't drop → non-linear site leakage. If resistance tanks → non-linear entanglement.
